# Part 2: Baseline model

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.baseline_features import (
    build_text_features,
    combine_with_metadata,
    train_logreg,
    evaluate_model
)
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

In [2]:
# Part 2 - Data paths 
TRAIN_PATH = '../data/processed/news_stratified_train.csv'
VAL_PATH = '../data/processed/news_stratified_val.csv'

# Load content for optional metadata features if available
cols_to_keep = ['content_processed', 'content', 'type']

train_df = pd.read_csv(TRAIN_PATH, low_memory=False, usecols=cols_to_keep, on_bad_lines='skip')
val_df = pd.read_csv(VAL_PATH, low_memory=False, usecols=cols_to_keep, on_bad_lines='skip')

# Basic cleaning
train_df = train_df.dropna(subset=['content_processed', 'type']).copy()
val_df = val_df.dropna(subset=['content_processed', 'type']).copy()

train_df['type'] = train_df['type'].astype(int)
val_df['type'] = val_df['type'].astype(int)

print(f"Train: {train_df.shape}, Val: {val_df.shape}")
print(f"\nTrain label distribution:\n{train_df['type'].value_counts()}")
print(f"\nVal label distribution:\n{val_df['type'].value_counts()}")


Train: (757768, 3), Val: (94721, 3)

Train label distribution:
type
1    427304
0    330464
Name: count, dtype: int64

Val label distribution:
type
1    53413
0    41308
Name: count, dtype: int64


We load the stratified train and validation splits from the full 995K dataset. The splits follow an 80/10/10 ratio with preserved class balance across labels.

## Feature Extraction
We use CountVectorizer with the top 10,000 most frequent words as text features. We also build a metadata feature matrix with URL count, number count, and article length for comparison in Task 3.

In [3]:
# Part 2 - Build feature matrices
X_train, X_val, vectorizer = build_text_features(
    train_df['content_processed'],
    val_df['content_processed'],
    max_features=10000
)

y_train = train_df['type'].values
y_val = val_df['type'].values

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")

X_train shape: (757768, 10000)
X_val shape: (94721, 10000)


In [4]:
# Optional metadata matrices for Task 3
X_train_meta = combine_with_metadata(X_train, train_df)
X_val_meta = combine_with_metadata(X_val, val_df)

print(f"X_train + metadata shape: {X_train_meta.shape}")
print(f"X_val + metadata shape: {X_val_meta.shape}")

X_train + metadata shape: (757768, 10003)
X_val + metadata shape: (94721, 10003)


## Logistic Regression Baseline
We train a logistic regression classifier on text features only. We set class_weight='balanced' to handle class imbalance in the training data.

In [5]:
# Part 2 - Train logistic regression baseline
model = train_logreg(
    X_train,
    y_train,
    C=1.0,
    max_iter=100,
    solver='liblinear',
    class_weight='balanced'
)

print("Model trained.")
print(f"Hyperparameters: C={model.C}, max_iter={model.max_iter}, solver={model.solver}, class_weight={model.class_weight}")

Model trained.
Hyperparameters: C=1.0, max_iter=100, solver=liblinear, class_weight=balanced


## Evaluation on Validation Set
We evaluate the model on the validation set using macro F1-score and a full classification report.

In [6]:
# Part 2 - Evaluate baseline on validation set
results = evaluate_model(model, X_val, y_val, label_names=['reliable', 'fake'])

print(results['report'])
print(f"Macro F1: {results['macro_f1']:.4f}")
print(f"Binary F1: {results['binary_f1']:.4f}")
print("\nConfusion matrix:")
print(results['confusion_matrix'])

              precision    recall  f1-score   support

    reliable       0.85      0.83      0.84     41308
        fake       0.87      0.89      0.88     53413

    accuracy                           0.86     94721
   macro avg       0.86      0.86      0.86     94721
weighted avg       0.86      0.86      0.86     94721

Macro F1: 0.8579
Binary F1: 0.8779

Confusion matrix:
[[34104  7204]
 [ 5987 47426]]


## Feature Analysis
We inspect the most influential words for each class to understand what the model has learned.

In [7]:
import numpy as np

# Top features for each class
feature_names = vectorizer.get_feature_names_out()

top_n = 20
fake_coefs = model.coef_[0]

top_fake = np.argsort(fake_coefs)[-top_n:][::-1]
top_reliable = np.argsort(fake_coefs)[:top_n]

print("Top 20 words predicting FAKE:")
for idx in top_fake:
    print(f"  {feature_names[idx]}: {fake_coefs[idx]:.3f}")

print("\nTop 20 words predicting RELIABLE:")
for idx in top_reliable:
    print(f"  {feature_names[idx]}: {fake_coefs[idx]:.3f}")

Top 20 words predicting FAKE:
  sputnik: 2.703
  conservapedia: 2.566
  lifezett: 2.390
  abovetopsecretcom: 2.370
  blockchain: 2.220
  eowyn: 2.201
  rondeau: 2.121
  tlb: 1.974
  stumbleupon: 1.966
  slideshow: 1.900
  naturalnew: 1.875
  iwb: 1.745
  investwatchblog: 1.732
  blocker: 1.670
  psnumtokenmillion: 1.589
  lifesitenew: 1.451
  sheepl: 1.425
  at: 1.321
  wnd: 1.296
  predefin: 1.258

Top 20 words predicting RELIABLE:
  pagesnumtoken: -2.960
  misstat: -2.324
  diari: -1.839
  shorefront: -1.834
  kossack: -1.807
  kos: -1.533
  comer: -1.449
  nytimescom: -1.357
  dkos: -1.238
  feedback: -1.225
  timess: -1.198
  login: -1.176
  misspel: -1.135
  rec: -1.101
  ist: -1.083
  dailyko: -1.035
  adblock: -0.978
  nov: -0.897
  nyt: -0.871
  huffpost: -0.844


The model primarily learns domain-specific patterns rather than linguistic ones. The top words predicting fake are domain names and web artefacts such as `sputnik`, `conservapedia`, and `abovetopsecret`, while the top words predicting reliable are domain names such as `nytimes` and `dailyko`. This suggests the model has learned to recognise sources rather than truth indicators.

## Summary
The baseline model achieves a macro F1 of 0.8579 on the validation set using text features only. Fake articles are classified slightly better (F1=0.88) than reliable articles (F1=0.84).

## Metadata Feature Comparison
We train a second model with metadata features added to evaluate whether URL count, number count, and article length improve classification performance.

In [8]:
# Train and evaluate model with metadata features
model_meta = train_logreg(
    X_train_meta,
    y_train,
    C=1.0,
    max_iter=100,
    solver='liblinear',
    class_weight='balanced'
)

results_meta = evaluate_model(model_meta, X_val_meta, y_val, label_names=['reliable', 'fake'])

print("Text only:")
print(f"  Macro F1: {results['macro_f1']:.4f}")
print("\nText + metadata:")
print(f"  Macro F1: {results_meta['macro_f1']:.4f}")
print("\nFull report (text + metadata):")
print(results_meta['report'])

Text only:
  Macro F1: 0.8579

Text + metadata:
  Macro F1: 0.8565

Full report (text + metadata):
              precision    recall  f1-score   support

    reliable       0.85      0.82      0.84     41308
        fake       0.86      0.89      0.88     53413

    accuracy                           0.86     94721
   macro avg       0.86      0.85      0.86     94721
weighted avg       0.86      0.86      0.86     94721



Adding metadata features (URL count, number count, article length) does not improve performance, the macro F1 drops marginally from 0.8579 to 0.8565. For the remainder of the project we use the text-only model.

## Save Model
We save the trained model and vectorizer for use in evaluation.

In [9]:
import joblib

joblib.dump(model, '../models/baseline_model.joblib')
joblib.dump(vectorizer, '../models/baseline_vectorizer.joblib')

print("Model and vectorizer saved.")

Model and vectorizer saved.
